<a href="https://colab.research.google.com/github/Bruno-Paulo/PETs/blob/main/Differential_Privacy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Differential Privacy Tutorial

_Last run: May 1, 2025_

## 1. Introduction

This tutorial explores Differential Privacy using the [Opacus](https://github.com/pytorch/opacus) library from PyTorch. You will learn how to train privacy-preserving machine learning models, ensuring sensitive data remains protected while still enabling valuable insights.



### 1.1 Scenario

LifeMed Analytics, a healthcare technology company, is developing machine learning models to predict stroke risk based on patient data. However, because this data contains sensitive medical information, sharing it for research or collaboration poses significant **privacy risks** and could lead to regulatory **non-compliance** with laws such as GDPR and HIPAA.  

To address these concerns, LifeMed needs a way to train models while ensuring that **individual patient data remains protected**.


### 1.2 Solution

Differential privacy provides a solution by adding controlled noise to the training process, preventing the extraction of specific patient details, while still allowing the model to learn useful patterns.

To balance privacy and utility, differential privacy is applied during model training. This allows LifeMed to:

- Protect patient confidentiality while enabling effective predictive modelling.
- Protect against data breaches by preventing the extraction of individual data.
- Ensure compliance with privacy regulations while maintaining the utility of the dataset.

### 1.3 Differential Privacy

Differential Privacy (DP) is a technique that protects individual data while allowing useful insights to be gained from data sets. It works by adding **noise** to the data or query results, making it difficult to determine whether a particular individual's data is included.  

There are two main types:  

- **Global Differential Privacy (GDP):** Noise is added after the data has been aggregated.  
- **Local Differential Privacy (LDP):** Noise is added to each individual record before it is shared.  

DP is widely used in data analytics, machine learning and privacy-preserving applications, helping organisations comply with regulations such as GDPR and HIPAA while maintaining the utility of the data.



### 1.4 Outline of this tutorial

By the end of this tutorial, you will understand how to incorporate differential privacy into machine learning models. You will learn how to balance privacy with model utility and how to compare the performance of differential privacy models with non-private models.

**Training a Differential Privacy Model**

We will start by building a machine learning model for stroke prediction and applying differential privacy during training. This ensures that individual data points in the dataset remain protected, while still allowing the model to learn meaningful patterns.

**Evaluating the effect of differential privacy on model performance**

Once training is complete, we will evaluate the differential privacy model and compare it to a non-private counterpart. This will highlight the trade-offs between privacy guarantees and model accuracy.

**Understanding and tuning privacy parameters**

Finally, we will explore key hyperparameters such as noise multiplier, batch size and gradient clipping. By adjusting these, we can fine-tune the balance between privacy and model utility.



## 2. Setup

In order to apply differential privacy we will need to download the original data and then configure the Python library that will assist in the tutorial.


---


***Note:*** This code will only work correctly if you use the Google
Chrome browser.



---





### 2.1 Downloading the original dataset

Please download the healthcare dataset in CSV file format from [here](https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset).

Now we need attach the CSVs to this Google Colab notebook. Run the code below and then Click the "Choose Files" button. In the file picker choose the CSV file that you have just downloaded.


In [ ]:
from google.colab import files

# Optional: You can skip this step if you are running the code on your own
# machine
uploaded = files.upload()

Saving healthcare-dataset-stroke-data.csv to healthcare-dataset-stroke-data.csv


The csv files are now available in the folder `content/`.

### 2.2 Installing the Opacus library

Opacus is a PyTorch library designed to efficiently integrate differential privacy into deep learning models. It enables training with minimal code changes while maintaining performance and scalability. Key features:

- Efficient privacy preserving training with minimal overhead.

- Automatic privacy accounting to track the level of protection.

- Seamless integration with PyTorch, requiring only a few additional steps.


If you need more help [Browse all tutorials](https://opacus.ai/tutorials/) or visit the [full documentation](https://opacus.ai/).

To install the Opacus library, run the following:




In [ ]:
%pip install opacus

### 2.3 Importing libraries

Now that we have all dependencies installed, we can import everything we need for this tutorial.



---
***Note***: If you get an error, try restarting the execution time on the "Execution Time" tab.


---




In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from opacus import PrivacyEngine
from opacus.validators import ModuleValidator
from opacus.utils.batch_memory_manager import BatchMemoryManager
from statistics import mean
from imblearn.over_sampling import SMOTE
import pandas as pd
import warnings

warnings.simplefilter("ignore")

DEVICE = DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"  # Try "cuda" to train on GPU
print(f"Training on {DEVICE}")
print(f"PyTorch {torch.__version__}")

Training on cpu
PyTorch 2.6.0+cu124




---
***Note***

It is possible to switch to a runtime that has GPU acceleration enabled (on Google Colab: `Runtime > Change runtime type > Hardware accelerator: GPU > Save`). However, that Google Colab is not always able to offer GPU acceleration. If you see an error related to GPU availability in one of the following sections, consider switching back to CPU-based execution by setting `DEVICE = torch.device("cpu")`. If the runtime has GPU acceleration enabled, you should see the output `Training on cuda`, otherwise it'll say `Training on cpu`.



---



## 3. Operations on Data



---



***Note:*** This introduction assumes basic familiarity with PyTorch, so it doesn't cover the PyTorch-related aspects in full detail. If you want to dive deeper into PyTorch, we recommend [*DEEP LEARNING WITH PYTORCH: A 60 MINUTE BLITZ*](https://pytorch.org/tutorials/beginner/deep_learning_60min_blitz.html).



---

### 3.1 Pre-process the data

We will create the load_data function which will load and pre-process the data. This process is divided into smaller steps and functions.




#### 3.1.1 Handle missing values and drop unnecessary columns

We clean the dataset by handling missing values and removing irrelevant columns.

In [ ]:
def clean_data(df):
    df = df.drop(columns=["id"])  # Drop irrelevant columns
    df["bmi"] = df["bmi"].fillna(df["bmi"].median())  # Fill missing BMI values
    return df

#### 3.1.2 Split features and target

We separate the input features (X) from the target variable (y).

In [ ]:
def split_features_target(df):
    X = df.drop(columns=["stroke"])
    y = df["stroke"]
    return X, y

#### 3.1.3 Define categorical and numerical features

We define transformations for numerical and categorical columns.

In [ ]:
def get_feature_transformers(X):
    """Create preprocessing pipelines for numerical and categorical features."""
    categorical_features = X.select_dtypes(include=["object"]).columns
    numeric_features = X.select_dtypes(include=["float64", "int64"]).columns

    numeric_transformer = Pipeline(steps=[("scaler", StandardScaler())])
    categorical_transformer = Pipeline(steps=[("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )
    return preprocessor

#### 3.1.4 Preprocess categorical and numerical features

We apply the previously defined transformations.

In [ ]:
def preprocess_features(X):
    """Apply transformations to numerical and categorical features."""
    preprocessor = get_feature_transformers(X)
    X_transformed = preprocessor.fit_transform(X)
    return pd.DataFrame(X_transformed)  # Convert back to DataFrame for compatibility with SMOTE

#### 3.1.5 Handle class imbalance with SMOTE

SMOTE is used to balance the dataset.

In [ ]:
def balance_classes(X, y):
    return SMOTE().fit_resample(X, y)

#### 3.1.6 Split data into train, validation, and test sets

We split the dataset into training, validation, and testing sets.

In [ ]:
def split_data(X, y, test_size=0.3, val_size=0.5, random_state=42):
    """Split the dataset into train, validation, and test sets."""
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=test_size, random_state=random_state)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=val_size, random_state=random_state)
    return X_train, X_val, X_test, y_train, y_val, y_test

#### 3.1.7 Convert data to pyTorch datasets

We define a custom dataset class and create DataLoaders.

In [ ]:
# Custom Dataset class
class CustomDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def create_dataloaders(X_train, X_val, X_test, y_train, y_val, y_test):
    """Convert data into PyTorch datasets and create DataLoaders."""
    train_dataset = CustomDataset(X_train.values, y_train)
    val_dataset = CustomDataset(X_val.values, y_val)
    test_dataset = CustomDataset(X_test.values, y_test)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    return train_loader, val_loader, test_loader

#### 3.1.8 Load and process data using these functions

Now we call all the above functions in sequence to load and process the data.

In [ ]:
def load_data():
    """Load, preprocess, balance, and split data, then return DataLoaders."""
    df = pd.read_csv("/content/healthcare-dataset-stroke-data.csv")
    df = clean_data(df)
    X, y = split_features_target(df)
    X = preprocess_features(X)
    X_resampled, y_resampled = balance_classes(X, y)
    X_train, X_val, X_test, y_train, y_val, y_test = split_data(X_resampled, y_resampled)
    return create_dataloaders(X_train, X_val, X_test, y_train, y_val, y_test)

### 3.2 Defining the model

We define our **StrokePredictionModel** along with training and testing functions to evaluate how well a trained model performs in predicting strokes. The aim is to compare the performance of a model trained on the original data with a model trained on differentially private data, to assess the impact of using differential privacy for machine learning tasks.

#### 3.2.1 Stroke prediction model class

In [ ]:
class StrokePredictionModel(nn.Module):
    def __init__(self, input_dim: int):
        super(StrokePredictionModel, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.5)
        )
        self.layer2 = nn.Sequential(
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.5)
        )
        self.output = nn.Linear(64, 1)

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.output(x)
        return x

#### 3.2.2 Train function

In [ ]:
def train(model, train_loader, optimizer, criterion, privacy_engine, epochs, device):
    model.to(device)
    model.train()
    for epoch in range(epochs):
        accs = []
        losses = []
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            outputs = model(X_batch).squeeze()
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            preds = (outputs > 0.5).float()  # Convert probabilities to binary predictions
            batch_accuracy = (preds == y_batch).float().mean().item()
            accs.append(batch_accuracy)
            losses.append(loss.item())

        # Report results
        printstr = (
        f"\t Epoch {epoch}. Accuracy: {mean(accs):.6f} | Loss: {mean(losses):.6f}"
        )
        if privacy_engine:
            epsilon = privacy_engine.get_epsilon(delta)
            printstr += f" | (ε = {epsilon:.2f}, δ = {delta})"

        print(printstr)

#### 3.2.3 Test function

In [ ]:
def test(model, test_loader, privacy_engine, device):
    model.to(device)
    model.eval()
    criterion = nn.BCEWithLogitsLoss()
    accs = []
    losses = []

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch).squeeze()
            loss = criterion(outputs, y_batch)

            preds = (outputs > 0.5).float()
            batch_accuracy = (preds == y_batch).float().mean().item()

            accs.append(batch_accuracy)
            losses.append(loss.item())

    mean_loss = mean(losses)
    mean_acc = mean(accs)

    printstr = (
        "\n----------------------------\n"
        f"Test Loss: {mean_loss:.6f} | Test Accuracy: {mean_acc:.6f}"
    )
    if privacy_engine:
        epsilon = privacy_engine.get_epsilon(delta)
        printstr += f" (ε = {epsilon:.2f}, δ = {delta})"
    print(printstr + "\n----------------------------\n")

### 3.3 Differential Privacy with Opacus

Differential privacy in Opacus is applied during the **model training process**, not to the dataset. Opacus modifies gradient computations to ensure privacy.

#### 3.3.1 Hyper-parameters

When training with Opacus, there are two sets of hyper-parameters to consider:

**General training hyper-parameters**
  - **Learning rate** (`learning_rate`): The step size for the optimiser.
    - A higher value makes the updates more significant, but noisy gradients often require a lower learning rate to stabilise the training.  

  - **Batch size** (`BATCH_SIZE`): Controls the number of examples processed in one step.
    - Larger batch sizes typically help convergence and reduce the relative impact of DP noise but increase the privacy cost (`EPSILON`). Opacus supports *virtual batches* to manage memory constraints while using logically larger batches.

  - **Epochs** (`EPOCHS`): Number of complete passes through the dataset.
    - For datasets where privacy noise significantly affects gradient updates, more epochs might help the model converge better. However, longer training can lead to diminishing returns, especially if privacy constraints limit gradient accuracy.

  
**Privacy-specific hyper-parameters**
  - **Max grad norm** (`MAX_GRAD_NORM`): Limits the L2 norm of the per-sample gradients to mitigate the effect of outliers.
    - A lower value limits extreme gradients, but may truncate meaningful information, affecting model performance.    

  - **Noise multiplier** (`noise_multiplier`): Controls the amount of noise added to gradients for privacy.  
    - Higher values provide stronger privacy guarantees (`EPSILON`), but reduce the utility of the model by adding more noise.    

  - **Delta** (`delta`): A very small probability that limits the likelihood of privacy guarantees failing.
    - A rule of thumb is to set it to be less than the inverse of the training data size (i.e., the population size).

  - **Epsilon** (`EPSILON`): Quantifies the privacy guarantee.
    - A smaller value represents stronger privacy. However, achieving a low (`EPSILON`) often requires higher noise, which may degrade model performance.



In [ ]:
# Training hyper-parameters
EPOCHS = 20
LEARNING_RATE = 0.15

# Privacy engine hyper-parameters
MAX_GRAD_NORM = 1.5
# The Noise Multiplier will be discussed ahead
delta = 1e-4
EPSILON = 10

Here is a summary of key hyper-parameters and suggested ranges:

| Parameter | Description | Suggested Range |
| --- | --- | --- |
| Max Grad Norm | Caps the L2 norm of gradients | 0.5 to 10 |
| Noise Multiplier | Controls noise added for privacy | ≥ 0.3 for strong DP |
| Delta | Probability of failure for DP guarantee | < 1/(dataset size) |
| Epsilon | Quantifies the privacy guarantee | < 10 for moderate privacy, < 1 for strong privacy|



---
***Note***

Privacy guarantees and model performance depend on the interaction of several parameters:  

- **Batch size vs. noise multiplier**:  
  - Larger batches reduce the relative noise added, which improves convergence.  
  - However, larger batches increase the cost of privacy (`EPSILON`).

- **Noise multiplier vs. max grade norm**:  
  - Increasing the `MAX_GRAD_NORM` allows more significant gradients to pass, but increases the added noise.
  - Tuning these together ensures effective learning while maintaining privacy.  


---




#### 3.3.2 Training with a memory manager

There's another constraint we should be aware of: **memory**.

To balance the peak memory requirement, which is proportional to `batch_size^2`, and training performance, we will use the `BatchMemoryManager`. It separates the logical batch size from the physical batch size.

- **Batch size:** defines how often the model is updated and how much DP noise is added.
- **`max_physical_batch_size`:** defines how many samples we process at a time.

With the `BatchMemoryManager` you create your DataLoader with a logical batch size and then provide the memory manager with the maximum physical batch size.

In [ ]:
def train_with_memory_manager(model, train_loader, optimizer, criterion, privacy_engine, epochs, device):
    model.to(device)
    model.train()
    for epoch in range(epochs):
        accs = []
        losses = []
        with BatchMemoryManager(
            data_loader=train_loader,
            max_physical_batch_size=128,
            optimizer=optimizer,
        ) as memory_safe_loader:
            for X_batch, y_batch in memory_safe_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)

                optimizer.zero_grad()
                outputs = model(X_batch).squeeze()
                loss = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()

                preds = (outputs > 0.5).float()  # Convert probabilities to binary predictions
                batch_accuracy = (preds == y_batch).float().mean().item()
                accs.append(batch_accuracy)
                losses.append(loss.item())

        # Report results
        printstr = (
        f"\t Epoch {epoch}. Accuracy: {mean(accs):.6f} | Loss: {mean(losses):.6f}"
        )
        if privacy_engine:
            epsilon = privacy_engine.get_epsilon(delta)
            printstr += f" | (ε = {epsilon:.2f}, δ = {delta})"

        print(printstr)

#### 3.3.3 Load the model

Now, we load and initialise the model.

In [ ]:
# Load data
train_loader, val_loader, test_loader = load_data()

# Initialize model
model = StrokePredictionModel(input_dim=train_loader.dataset[0][0].shape[0])

#### 3.3.4 Validate and fix the model

Opacus does not support all types of Pytorch layers. To check if your model is compatible with the privacy engine, there is a util class to validate your model called `ModuleValidator`.

[Now, let’s check if the model is compatible with Opacus.]: #

When you run the code below, you may be presented with a list of errors, indicating which modules are incompatible.

Recommended approach to deal with it is calling `ModuleValidator.fix(model)` - it tries to find the best replacement for incompatible modules.

In [ ]:
# Validate and fix the model
errors = ModuleValidator.validate(model, strict=False)
if errors:
    print(f"Model validation errors: {errors}")
    model = ModuleValidator.fix(model)
    print("Fixed model compatibility issues.")
ModuleValidator.validate(model, strict=False)

Model validation errors: [ShouldReplaceModuleError("BatchNorm cannot support training with differential privacy. The reason for it is that BatchNorm makes each sample's normalized value depend on its peers in a batch, ie the same sample x will get normalized to a different value depending on who else is on its batch. Privacy-wise, this means that we would have to put a privacy mechanism there too. While it can in principle be done, there are now multiple normalization layers that do not have this issue: LayerNorm, InstanceNorm and their generalization GroupNorm are all privacy-safe since they don't have this property.We offer utilities to automatically replace BatchNorms to GroupNorms and we will release pretrained models to help transition, such as GN-ResNet ie a ResNet using GroupNorm, pretrained on ImageNet"), ShouldReplaceModuleError("BatchNorm cannot support training with differential privacy. The reason for it is that BatchNorm makes each sample's normalized value depend on its p

[]

You can see, that after this, no exception is raised.

#### 3.3.5 Privacy engine, optimiser and loss criterion

We now proceed to instantiate the objects (privacy engine, model and optimizer) for our differentially-private training.

The purpose of the `PrivacyEngine` is to:

1. **Protect individual records:** By ensuring that the contribution of any single data point to the model’s updates is bounded and noise-added, making it difficult to infer the inclusion of a specific record in the dataset.
2. **Track privacy guarantees:** It tracks and calculates the privacy budget (`EPSILON`) consumed during training.


In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE)

# Attach PrivacyEngine
privacy_engine = PrivacyEngine()

model, optimizer, train_loader = privacy_engine.make_private_with_epsilon(
    module=model,
    optimizer=optimizer,
    data_loader=train_loader,
    max_grad_norm=MAX_GRAD_NORM,
    target_delta=delta,
    target_epsilon=EPSILON,
    epochs=EPOCHS,
)

#### 3.3.6 Noise multiplier

There is another parameter that we need to consider: `noise_multiplier`.

This governs the amount of noise added during training. Generally, more noise results in better privacy and lower utility.

By using the `make_private_with_epsilon`, the variable is calculated according to the other given hyperparameters such as `epsilon` and `delta`. Let's check the `noise_multiplier` in our model.

In [ ]:
print(f"Using noise_multiplier={optimizer.noise_multiplier} and max_grad_norm={MAX_GRAD_NORM}")

Using noise_multiplier=0.512237548828125 and max_grad_norm=1.5


### 3.4 Training the model

#### 3.4.1 Train the model with differential privacy

Finally, we can start training! We will be training for the number of epochs previously defined.

We will also benchmark this differentially-private model against a model without privacy and see the differences.

In [ ]:
# Train and test
## We will train with the memory manager function, but you can use the normal
## train function. It will output similar results.
train_with_memory_manager(model, train_loader, optimizer, criterion, privacy_engine, EPOCHS, DEVICE)

## train without the memory manager
#train(model, train_loader, optimizer, criterion, privacy_engine, EPOCHS, DEVICE)

test(model, test_loader, privacy_engine, DEVICE)

	 Epoch 0. Accuracy: 0.565509 | Loss: 0.712074 | (ε = 3.62, δ = 0.0001)
	 Epoch 1. Accuracy: 0.629859 | Loss: 0.762197 | (ε = 4.30, δ = 0.0001)
	 Epoch 2. Accuracy: 0.653263 | Loss: 0.786695 | (ε = 4.83, δ = 0.0001)
	 Epoch 3. Accuracy: 0.668965 | Loss: 0.786981 | (ε = 5.28, δ = 0.0001)
	 Epoch 4. Accuracy: 0.699559 | Loss: 0.722547 | (ε = 5.69, δ = 0.0001)
	 Epoch 5. Accuracy: 0.710923 | Loss: 0.701207 | (ε = 6.07, δ = 0.0001)
	 Epoch 6. Accuracy: 0.702885 | Loss: 0.748988 | (ε = 6.42, δ = 0.0001)
	 Epoch 7. Accuracy: 0.707139 | Loss: 0.749254 | (ε = 6.76, δ = 0.0001)
	 Epoch 8. Accuracy: 0.707311 | Loss: 0.767206 | (ε = 7.08, δ = 0.0001)
	 Epoch 9. Accuracy: 0.701650 | Loss: 0.739124 | (ε = 7.38, δ = 0.0001)
	 Epoch 10. Accuracy: 0.706160 | Loss: 0.741187 | (ε = 7.68, δ = 0.0001)
	 Epoch 11. Accuracy: 0.709062 | Loss: 0.744307 | (ε = 7.96, δ = 0.0001)
	 Epoch 12. Accuracy: 0.711654 | Loss: 0.753967 | (ε = 8.24, δ = 0.0001)
	 Epoch 13. Accuracy: 0.705234 | Loss: 0.744183 | (ε = 8.51, 

The differentially private model achieves a test accuracy of 78% with an epsilon of just under 10. This shows that we can achieve a minimal loss of privacy with good accuracy.

Let's see if the accuracy improves with the non-private model.

#### 3.4.2 Train the model without differential privacy

For comparison, let's train a non-private model. By keeping parameters such as learning rate and batch size the same, we can evaluate how the performance of the differentially private model compares to the non-private model.

The `BatchMemoryManager` is only needed for memory efficient training with private optimisers, so it is not used here.


In [ ]:
# Train the model without privacy for comparison
print("\nTraining without privacy:")
criterion = nn.BCEWithLogitsLoss()
model_nodp = StrokePredictionModel(input_dim=train_loader.dataset[0][0].shape[0]).to(DEVICE)
optimizer_nodp = torch.optim.SGD(model_nodp.parameters(), lr=LEARNING_RATE)


print("Train stats: \n")
train(model_nodp, train_loader, optimizer_nodp, criterion, None, EPOCHS, DEVICE)
# Final test after all epochs
test(model_nodp, test_loader, None, device=DEVICE)


Training without privacy:
Train stats: 

	 Epoch 0. Accuracy: 0.724177 | Loss: 0.511932
	 Epoch 1. Accuracy: 0.747387 | Loss: 0.479312
	 Epoch 2. Accuracy: 0.759149 | Loss: 0.474580
	 Epoch 3. Accuracy: 0.773719 | Loss: 0.462903
	 Epoch 4. Accuracy: 0.776032 | Loss: 0.464322
	 Epoch 5. Accuracy: 0.771542 | Loss: 0.459241
	 Epoch 6. Accuracy: 0.776670 | Loss: 0.450351
	 Epoch 7. Accuracy: 0.779031 | Loss: 0.450117
	 Epoch 8. Accuracy: 0.787573 | Loss: 0.439769
	 Epoch 9. Accuracy: 0.788540 | Loss: 0.445386
	 Epoch 10. Accuracy: 0.784818 | Loss: 0.434573
	 Epoch 11. Accuracy: 0.802621 | Loss: 0.419447
	 Epoch 12. Accuracy: 0.785805 | Loss: 0.428032
	 Epoch 13. Accuracy: 0.800653 | Loss: 0.414030
	 Epoch 14. Accuracy: 0.810388 | Loss: 0.404150
	 Epoch 15. Accuracy: 0.779736 | Loss: 0.435858
	 Epoch 16. Accuracy: 0.788512 | Loss: 0.428356
	 Epoch 17. Accuracy: 0.801068 | Loss: 0.413825
	 Epoch 18. Accuracy: 0.803122 | Loss: 0.411131
	 Epoch 19. Accuracy: 0.795385 | Loss: 0.427577

-------

#### 3.4.3 Comparing the results

The non-private model achieves slightly better accuracy and lower loss than the private model, demonstrating that training with Opacus introduces minimal overhead. While the private model achieves 78% accuracy with an epsilon of just under 10, the non-private model slightly outperforms it.

This comparison highlights the trade-off between privacy and performance.

## 4. Challenge: Strengthening Privacy

Imagine that **LifeMed Analytics** now needs to comply with stricter privacy regulations because it is handling even more sensitive healthcare data. The company decides to implement a stronger privacy guarantee with **epsilon = 1**. The company decides that a private model with ≥70% accuracy and loss ≤1 are good parameters for its model predictions.

The solutions can be found at the end of the tutorial.


**Task**

Adjust the parameters in **Hyper-parameters section (3.3.1)** to achieve the best possible accuracy (≥70%) and lowest loss (≤1) while maintaining a privacy guarantee of **EPSILON = 1**.
Key Parameters to Tune:

- **Learning rate**: start with a small learning rate and adjust it gradually to find the right balance between performance and privacy.  

- **Max grade norm (`MAX_GRAD_NORM`)**: start with 1.0. Explore the range [0.5, 10] to find the best setting for your model and dataset.  

- **Batch size**: larger batch sizes typically improve stability but may require tuning other parameters.

- **Epochs**: start with 10–20; increase for better accuracy at the cost of time and potential overfitting.


***Hints***:

Redo **Section 3.3**

1. Change the hyperparameters.
2. Load the model.
3. Validate and fix the model.
4. Create the privacy engine.
5. Finally, train and test the model. You will get full points if the model achieves an accuracy ≥ 70% and a loss ≤1 for **EPSILON = 1**.


**Instructions to submit the code**

1. Write your code on the code block below and test it out.
2. Save it as txt file.
3. Submit it on the [Differential Privacy forms](https://forms.gle/VoXt5rkDrYTdPJq56).

In [ ]:
# First Write and test your code here


## 5. Conclusions

This tutorial has guided you through the implementation of differential privacy in machine learning using Opacus. We started by understanding the importance of privacy-preserving techniques, then trained a stroke prediction model with differential privacy, and finally compared its performance with a non-private model.

Along the way, we explored key privacy parameters, such as noise multiplier and gradient clipping, to see how they affect both privacy guarantees and model accuracy. Our results showed that although differentially private models suffer some loss of performance, they still provide valuable insights while ensuring compliance with privacy regulations.

With this knowledge, you can now apply differential privacy techniques to your own machine learning projects, striking the right balance between privacy and utility.

## Solutions

### Solution for strengthening privacy

Please find a possible solution for `EPSILON` = 1, accuracy ≥ 70%, and loss ≤1.

#### 1. Hyper-parameters



In [ ]:
# Training hyper-parameters
EPOCHS = 20
LEARNING_RATE = 0.3

# Privacy engine hyper-parameters
MAX_GRAD_NORM = 1
# The Noise Multiplier will be discussed ahead
delta = 1e-4
EPSILON = 1

#### 2. Load the model

In [ ]:
# Load data
train_loader, val_loader, test_loader = load_data()

# Initialize model
model = StrokePredictionModel(input_dim=train_loader.dataset[0][0].shape[0])

#### 3. Validate and fix the model

In [ ]:
# Validate and fix the model
errors = ModuleValidator.validate(model, strict=False)
if errors:
    print(f"Model validation errors: {errors}")
    model = ModuleValidator.fix(model)
    print("Fixed model compatibility issues.")
ModuleValidator.validate(model, strict=False)

Model validation errors: [ShouldReplaceModuleError("BatchNorm cannot support training with differential privacy. The reason for it is that BatchNorm makes each sample's normalized value depend on its peers in a batch, ie the same sample x will get normalized to a different value depending on who else is on its batch. Privacy-wise, this means that we would have to put a privacy mechanism there too. While it can in principle be done, there are now multiple normalization layers that do not have this issue: LayerNorm, InstanceNorm and their generalization GroupNorm are all privacy-safe since they don't have this property.We offer utilities to automatically replace BatchNorms to GroupNorms and we will release pretrained models to help transition, such as GN-ResNet ie a ResNet using GroupNorm, pretrained on ImageNet"), ShouldReplaceModuleError("BatchNorm cannot support training with differential privacy. The reason for it is that BatchNorm makes each sample's normalized value depend on its p

[]

#### 4. Privacy engine

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE)

# Attach PrivacyEngine
privacy_engine = PrivacyEngine()

model, optimizer, train_loader = privacy_engine.make_private_with_epsilon(
    module=model,
    optimizer=optimizer,
    data_loader=train_loader,
    max_grad_norm=MAX_GRAD_NORM,
    target_delta=delta,
    target_epsilon=EPSILON,
    epochs=EPOCHS,
)

#### 5. Training and testing the model

In [ ]:
# Train and test
train_with_memory_manager(model, train_loader, optimizer, criterion, privacy_engine, EPOCHS, DEVICE)
test(model, test_loader, privacy_engine, DEVICE)

	 Epoch 0. Accuracy: 0.531120 | Loss: 0.869375 | (ε = 0.21, δ = 0.0001)
	 Epoch 1. Accuracy: 0.589143 | Loss: 0.912895 | (ε = 0.30, δ = 0.0001)
	 Epoch 2. Accuracy: 0.604430 | Loss: 0.921010 | (ε = 0.37, δ = 0.0001)
	 Epoch 3. Accuracy: 0.622841 | Loss: 0.962144 | (ε = 0.42, δ = 0.0001)
	 Epoch 4. Accuracy: 0.605471 | Loss: 0.979788 | (ε = 0.48, δ = 0.0001)
	 Epoch 5. Accuracy: 0.597592 | Loss: 0.935400 | (ε = 0.52, δ = 0.0001)
	 Epoch 6. Accuracy: 0.621776 | Loss: 0.941526 | (ε = 0.57, δ = 0.0001)
	 Epoch 7. Accuracy: 0.644395 | Loss: 0.933645 | (ε = 0.61, δ = 0.0001)
	 Epoch 8. Accuracy: 0.603912 | Loss: 0.945011 | (ε = 0.65, δ = 0.0001)
	 Epoch 9. Accuracy: 0.607859 | Loss: 0.949480 | (ε = 0.68, δ = 0.0001)
	 Epoch 10. Accuracy: 0.619722 | Loss: 0.941184 | (ε = 0.72, δ = 0.0001)
	 Epoch 11. Accuracy: 0.624022 | Loss: 0.941186 | (ε = 0.75, δ = 0.0001)
	 Epoch 12. Accuracy: 0.648103 | Loss: 0.959332 | (ε = 0.79, δ = 0.0001)
	 Epoch 13. Accuracy: 0.636061 | Loss: 0.958115 | (ε = 0.82, 